## Setup

In [1]:
import os
from dotenv import load_dotenv
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from huggingface_hub import login

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Directory structure ───────────────────────────────────────────────────────
# Adjust DATA_DIR to where your .tsv/.csv files live
DATA_DIR       = Path('./data')
TALKDOWN_DIR   = Path('talkdown')          # see Block 3 for download instructions
CHECKPOINTS    = Path('checkpoints')
CHECKPOINTS.mkdir(exist_ok=True)

TALKDOWN_CKPT  = CHECKPOINTS / 'deberta_talkdown'
PCL_CKPT       = CHECKPOINTS / 'deberta_pcl_best'

# HF
load_dotenv()  # loads from .env in current directory
HF_TOKEN = os.getenv('HF_TOKEN')
assert HF_TOKEN is not None, '.env file missing or HF_TOKEN not set!'
login(token=HF_TOKEN)
print('HuggingFace authentication successful.')

print('Directories ready.')
print(f'TalkDown checkpoint path : {TALKDOWN_CKPT}')
print(f'PCL checkpoint path      : {PCL_CKPT}')

Device: cuda
GPU: Tesla T4


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace authentication successful.
Directories ready.
TalkDown checkpoint path : checkpoints/deberta_talkdown
PCL checkpoint path      : checkpoints/deberta_pcl_best


## Data loading & Preprocess

In [2]:
import html, re

def clean_text(text) -> str:
    """Unescape HTML entities and strip residual tags. Handles NaN."""
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    return text.strip()

def add_keyword_prefix(row) -> str:
    """Prepend topic keyword as a bracketed token."""
    return f"[{row['keyword']}] {row['text']}"

# ── Load raw TSV ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv(
    DATA_DIR / 'dontpatronizeme_pcl.tsv', sep='\t', header=None,
    names=['par_id','art_id','keyword','country','text','label']
)

# Drop disclaimer / separator rows
df_raw = df_raw[pd.to_numeric(df_raw['par_id'], errors='coerce').notna()].copy()
df_raw['par_id'] = df_raw['par_id'].astype(int)

# ── Load split IDs ────────────────────────────────────────────────────────────
train_ids = pd.read_csv(DATA_DIR / 'train_semeval_parids-labels.csv')
dev_ids   = pd.read_csv(DATA_DIR / 'dev_semeval_parids-labels.csv')

train = df_raw[df_raw['par_id'].isin(train_ids['par_id'])].copy()
dev   = df_raw[df_raw['par_id'].isin(dev_ids['par_id'])].copy()

# ── Binarise labels (0-4 scale → 0/1) ────────────────────────────────────────
train['label'] = (train['label'].astype(int) >= 1).astype(int)
dev['label']   = (dev['label'].astype(int) >= 1).astype(int)

# ── Clean text ────────────────────────────────────────────────────────────────
train['text'] = train['text'].apply(clean_text)
dev['text']   = dev['text'].apply(clean_text)

# ── Add keyword prefix ────────────────────────────────────────────────────────
train['text_input'] = train.apply(add_keyword_prefix, axis=1)
dev['text_input']   = dev.apply(add_keyword_prefix, axis=1)

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f'Train size : {len(train)}')
print(f'Dev size   : {len(dev)}')
print(f'Train label distribution:\n{train["label"].value_counts()}')
print(f'Dev label distribution:\n{dev["label"].value_counts()}')
print(f'\nSample input (train[0]):')
print(train['text_input'].iloc[0])
assert train['text'].isna().sum() == 0, 'NaN texts in train!'
assert dev['text'].isna().sum() == 0,   'NaN texts in dev!'
assert set(train['label'].unique()) == {0, 1}, 'Labels not binarised!'
print('\nAll sanity checks passed.')

Train size : 8375
Dev size   : 2094
Train label distribution:
label
0    6825
1    1550
Name: count, dtype: int64
Dev label distribution:
label
0    1704
1     390
Name: count, dtype: int64

Sample input (train[0]):
[hopeless] We 're living in times of absolute insanity , as I 'm pretty sure most people are aware . For a while , waking up every day to check the news seemed to carry with it the same feeling of panic and dread that action heroes probably face when they 're trying to decide whether to cut the blue or green wire on a ticking bomb -- except the bomb 's instructions long ago burned in a fire and imminent catastrophe seems the likeliest outcome . It 's hard to stay that on-edge for that long , though , so it 's natural for people to become inured to this constant chaos , to slump into a malaise of hopelessness and pessimism .

All sanity checks passed.


In [3]:
test = pd.read_csv(
    DATA_DIR / 'task4_test.tsv', sep='\t', header=None,
    names=['par_id', 'art_id', 'keyword', 'country', 'text']
)

# No disclaimer rows to filter — IDs are already clean (t_0, t_1, ...)
test['text'] = test['text'].apply(clean_text)
test['text_input'] = test.apply(add_keyword_prefix, axis=1)

print(f'Test size: {len(test)}')
print(f'Sample test input: {test["text_input"].iloc[0]}')
assert len(test) == 3832, f'Expected 3832 test rows, got {len(test)}'
assert test['text'].isna().sum() == 0, 'NaN texts in test!'
print('Test set loaded successfully.')

Test size: 3832
Sample test input: [vulnerable] In the meantime , conservatives are working to weaken Clinton and drive down her numbers in early voting states , where she is increasingly vulnerable . They are , in effect , doing Sanders 's dirty work for him while he avoids scrutiny .
Test set loaded successfully.


## Talkdown dataset

In [4]:
import urllib.request
import tarfile

# Download and extract TalkDown dataset
TALKDOWN_URL = 'https://cs.stanford.edu/people/zijwang/talkdown/talkdown.tar.gz'
TALKDOWN_TAR = Path('talkdown.tar.gz')

# ── Skip if already downloaded ────────────────────────────────────────────────
if (TALKDOWN_DIR / 'balanced_train.jsonl').exists():
    print('TalkDown already extracted — skipping download.')
else:
    TALKDOWN_DIR.mkdir(exist_ok=True)

    # Download
    print(f'Downloading TalkDown from {TALKDOWN_URL}...')
    urllib.request.urlretrieve(TALKDOWN_URL, TALKDOWN_TAR)
    print(f'Downloaded: {TALKDOWN_TAR} ({TALKDOWN_TAR.stat().st_size / 1e6:.1f} MB)')

    # Extract
    print('Extracting...')
    with tarfile.open(TALKDOWN_TAR, 'r:gz') as tar:
        tar.extractall(path=TALKDOWN_DIR)
    print(f'Extracted to {TALKDOWN_DIR}/')

    # Clean up tar file
    TALKDOWN_TAR.unlink()
    print('Removed tar file.')

    # Verify
    expected = ['balanced_train.jsonl', 'balanced_dev.jsonl', 'balanced_test.jsonl']
    for f in expected:
        # Search recursively in case tar extracted into a subdirectory
        matches = list(TALKDOWN_DIR.rglob(f))
        assert len(matches) > 0, f'{f} not found after extraction!'
        print(f'  ✓ {matches[0]}')

    print('TalkDown download and extraction complete.')

# ── Move files to talkdown/ root if extracted into subdirectory ───────────────
import shutil
for jsonl in TALKDOWN_DIR.rglob('*.jsonl'):
    target = TALKDOWN_DIR / jsonl.name
    if jsonl != target:
        shutil.move(str(jsonl), str(target))
        print(f'Moved {jsonl.name} to {target}')

print(f'Files in {TALKDOWN_DIR}/:')
for f in sorted(TALKDOWN_DIR.iterdir()):
    print(f'  {f.name}')

TalkDown already extracted — skipping download.
Files in talkdown/:
  .ipynb_checkpoints
  README.md
  annotated.jsonl
  balanced_dev.jsonl
  balanced_test.jsonl
  balanced_train.jsonl
  imbalanced_dev.jsonl
  imbalanced_test.jsonl
  imbalanced_train.jsonl


In [5]:
import json

def load_jsonl(path):
    records = []
    with open(path, 'r') as f:
        for line in f:
            records.append(json.loads(line.strip()))
    return pd.DataFrame(records)

# assert files exist
assert (TALKDOWN_DIR / 'balanced_train.jsonl').exists(), 'TalkDown train file not found!'
assert (TALKDOWN_DIR / 'balanced_dev.jsonl').exists(), 'TalkDown dev file not found!'

td_train = load_jsonl(TALKDOWN_DIR / 'balanced_train.jsonl')
td_val   = load_jsonl(TALKDOWN_DIR / 'balanced_dev.jsonl')

# Use 'post' as text, convert bool label to int
td_train['text']  = td_train['post'].apply(clean_text)
td_train['label'] = td_train['label'].astype(int)
td_val['text']    = td_val['post'].apply(clean_text)
td_val['label']   = td_val['label'].astype(int)

td_train = td_train.dropna(subset=['text']).reset_index(drop=True)
td_val   = td_val.dropna(subset=['text']).reset_index(drop=True)

print(f'TalkDown train: {len(td_train)} | val: {len(td_val)}')
print(f'Label distribution:\n{td_train["label"].value_counts()}')
print(f'Sample: {td_train["text"].iloc[0][:100]}')

TalkDown train: 5208 | val: 650
Label distribution:
label
1    2604
0    2604
Name: count, dtype: int64
Sample: Well a guy is saying Barra, who has those great credentials, sucks... It is just so rude an misinfor


## Tokeniser & classes

In [6]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=256):
        self.texts      = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.labels     = labels.tolist() if labels is not None else None
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation    = True,
            max_length    = self.max_length,
            padding       = 'max_length',
            return_tensors= 'pt'
        )
        item = {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze()
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ── Sanity check tokeniser ────────────────────────────────────────────────────
sample = tokenizer(
    '[homeless] This is a test sentence.',
    truncation=True, max_length=MAX_LENGTH, return_tensors='pt'
)
print(f'Tokeniser loaded: {MODEL_NAME}')
print(f'Sample token count: {sample["input_ids"].shape[1]}')
print('Tokeniser sanity check passed.')

Tokeniser loaded: microsoft/deberta-v3-base
Sample token count: 10
Tokeniser sanity check passed.


## Utilities

In [7]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report

def compute_f1(preds, labels, positive_label=1):
    return f1_score(labels, preds, pos_label=positive_label)

def evaluate(model, dataloader, device):
    """Returns F1 on positive class, all predictions, and all labels."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)
            outputs = model(**kwargs)
            preds   = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    f1 = compute_f1(all_preds, all_labels)
    return f1, all_preds, all_labels

def train_model(
    model, train_loader, val_loader,
    n_epochs, lr, warmup_steps,
    device, save_path,
    class_weights=None
):
    """
    Generic training loop. Saves best checkpoint by val F1.
    Skips training if checkpoint already exists.
    """
    if Path(save_path).exists():
        print(f'Checkpoint found at {save_path} — loading saved model.')
        model = AutoModelForSequenceClassification.from_pretrained(save_path)
        model.to(device)
        return model

    total_steps = len(train_loader) * n_epochs
    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps
    )

    if class_weights is not None:
        weights    = torch.tensor(class_weights, dtype=torch.float).to(device)
        loss_fn    = torch.nn.CrossEntropyLoss(weight=weights)
    else:
        loss_fn    = torch.nn.CrossEntropyLoss()

    best_f1 = 0.0
    model.to(device)

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for step, batch in enumerate(train_loader):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                kwargs['token_type_ids'] = batch['token_type_ids'].to(device)
            outputs = model(**kwargs)
            loss    = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            if step % 50 == 0:
                print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

        avg_loss = total_loss / len(train_loader)
        val_f1, _, _ = evaluate(model, val_loader, device)
        print(f'Epoch {epoch+1}/{n_epochs} | Avg Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}')

        if val_f1 > best_f1:
            best_f1 = val_f1
            model.save_pretrained(save_path)
            tokenizer.save_pretrained(save_path)
            print(f'  ✓ New best model saved (F1={best_f1:.4f})')

    print(f'Training complete. Best val F1: {best_f1:.4f}')
    return model

print('Training utilities loaded.')

Training utilities loaded.


## Stage 1: fine tune on TalkDown

In [8]:
if not (TALKDOWN_DIR / 'balanced_train.jsonl').exists():
    print('TalkDown dataset not found — skipping Stage 1.')
elif TALKDOWN_CKPT.exists():
    print(f'TalkDown checkpoint found — loading saved model, skipping training.')
    td_model = AutoModelForSequenceClassification.from_pretrained(
        TALKDOWN_CKPT,
        num_labels=2,
        torch_dtype=torch.float32
    )
    td_model.to(DEVICE)
    print(f'Stage 1 complete. Loaded from: {TALKDOWN_CKPT}')
else:
    # ── Clear GPU memory ─────────────────────────────────────────────────────
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # ── Filter out empty/null texts ──────────────────────────────────────────
    td_train = td_train[td_train['text'].str.strip().str.len() > 5].reset_index(drop=True)
    td_val   = td_val[td_val['text'].str.strip().str.len() > 5].reset_index(drop=True)
    print(f'TalkDown train after filtering: {len(td_train)}')
    print(f'TalkDown val after filtering  : {len(td_val)}')
    # ── Check for NaN labels ─────────────────────────────────────────────────
    assert td_train['label'].isna().sum() == 0, 'NaN labels in TalkDown train!'
    assert set(td_train['label'].unique()).issubset({0, 1}), f'Unexpected labels: {td_train["label"].unique()}'
    print(f'Label distribution:\n{td_train["label"].value_counts()}')
    BATCH_SIZE_TD = 8
    td_train_dataset = TextDataset(td_train['text'], td_train['label'], tokenizer, MAX_LENGTH)
    td_val_dataset   = TextDataset(td_val['text'],   td_val['label'],   tokenizer, MAX_LENGTH)
    td_train_loader  = DataLoader(td_train_dataset, batch_size=BATCH_SIZE_TD, shuffle=True, drop_last=True)
    td_val_loader    = DataLoader(td_val_dataset,   batch_size=BATCH_SIZE_TD, drop_last=False)
    # ── Sanity check: verify no NaN in first batch ───────────────────────────
    sample_batch = next(iter(td_train_loader))
    print(f'Batch input_ids shape : {sample_batch["input_ids"].shape}')
    print(f'Batch labels          : {sample_batch["labels"][:8]}')
    assert not torch.isnan(sample_batch['input_ids'].float()).any(), 'NaN in input_ids!'
    # ── Load fresh DeBERTa ────────────────────────────────────────────────────
    td_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, torch_dtype=torch.float32
    )
    for name, param in td_model.named_parameters():
        if torch.isnan(param).any():
            print(f'NaN in parameter: {name}')
            break
    else:
        print('Model parameters clean — no NaN detected.')
    print('Starting Stage 1: TalkDown fine-tuning...')
    td_model = train_model(
        model         = td_model,
        train_loader  = td_train_loader,
        val_loader    = td_val_loader,
        n_epochs      = 3,
        lr            = 5e-6,
        warmup_steps  = 200,
        device        = DEVICE,
        save_path     = TALKDOWN_CKPT,
        class_weights = None
    )
    assert TALKDOWN_CKPT.exists(), 'TalkDown checkpoint not saved!'
    print(f'Stage 1 complete. Checkpoint: {TALKDOWN_CKPT}')

`torch_dtype` is deprecated! Use `dtype` instead!


TalkDown checkpoint found — loading saved model, skipping training.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Stage 1 complete. Loaded from: checkpoints/deberta_talkdown
